In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML


dt=1e-3 ; m1=1 ; m2=1; g=9.8 ; l1=2 ; l2=2
theta01 = np.pi/2 ; theta02 = np.pi/2       #condiciones iniciales
N=int(1e6) 

In [152]:
#Funciones

def Cart(l1,l2,theta_1,theta_2):
  x1 = l1 * np.sin(theta_1)
  y1 = l1 * np.cos(theta_1)
  x2 = l2 * np.sin(theta_2) + x1
  y2 = l2 * np.cos(theta_2) + y1
  return x1, x2, y1, y2


def Ec_EuLag(theta_1,theta_2,w1,w2,l1,l2,m1,m2,g) :
    k=(m2*l2)/(m1+m2)
    A = np.array([
        [l1, k * np.cos(theta_2 - theta_1)],
        [l1 * np.cos(theta_2 - theta_1), l2]
    ])
    B = np.array([
        k * np.sin(theta_2 - theta_1) * (w2**2) + g * np.sin(theta_1),
        -l1 * np.sin(theta_2 - theta_1) * (w1**2) + g * np.sin(theta_2)
    ])
    X = np.linalg.solve(A, B)  #AX = B
    a1, a2 = X[0], X[1]
    
    return a1, a2

def Motion(t,theta_1,theta_2,w1,w2,a1,a2,l1,l2,m1,m2,g):
    for i in range(len(t)-1):
        dt = t[i+1] - t[i]
        # predictores de aceleración, velocidad y posición (Euler)
        a1_pred, a2_pred = Ec_EuLag(theta_1[i], theta_2[i], w1[i], w2[i], l1, l2, m1, m2, g)  # aceleración

        w1_pred = w1[i] + a1_pred * dt  # Euler vx(i+1)=vx(i)+a_pred*dt ;
        w2_pred = w2[i] + a2_pred * dt

        theta1_pred = theta_1[i] + w1[i] * dt  # Euler x(i+1)=x(i)+vx_pred*dt ;
        theta2_pred = theta_2[i] + w2[i] * dt

        # correctores de posición, aceleración y velocidad (HEUN)
        theta_1[i+1] = theta_1[i] + 0.5 * (w1[i] + w1_pred) * dt
        theta_2[i+1] = theta_2[i] + 0.5 * (w2[i] + w2_pred) * dt

        a1_next, a2_next = Ec_EuLag(theta1_pred, theta2_pred, w1_pred, w2_pred, l1, l2, m1, m2, g)

        w1[i+1] = w1[i] + 0.5 * (a1_pred + a1_next) * dt
        w2[i+1] = w2[i] + 0.5 * (a2_pred + a2_next) * dt

    return theta_1, theta_2, w1, w2, a1, a2

In [153]:
#Inicialización
r1, r2 = np.zeros((2, N)), np.zeros((2, N))
theta_1, theta_2, w1, w2, a1, a2 = (np.zeros(N) for _ in range(6))

r1[0,0] , r2[0,0] , r1[1,0] , r2[1,0] = Cart(l1,l2,theta01,theta02)
theta_1[0]=theta01
theta_2[0]=theta02
a1[0], a2[0] = Ec_EuLag(theta_1[0], theta_2[0], w1[0], w2[0], l1,l2,m1,m2,g)
t=np.linspace(0,dt*(N-1),N)


In [154]:
#Principal loop
theta_1,theta_2,w1,w2,a1,a2 =Motion(t,theta_1,theta_2,w1,w2,a1,a2,l1,l2,m1,m2,g)
r1[0,:] , r2[0,:] , r1[1,:] , r2[1,:] =Cart(l1,l2,theta_1,theta_2)

In [ ]:
# --- Parámetros de la animación ---
T_anim   = 20      # segundos simulados a animar (máx. dt*(N-1))
fps      = 30      # fotogramas por segundo
speed    = 1.0     # 1 = tiempo real, 0.5 = cámara lenta
trail_s  = 2.0     # duración de la estela de m2 (s); 0 = sin estela

# --- Submuestreo: un fotograma cada 'step' pasos de simulación ---
step   = max(1, round(speed / (fps * dt)))
n_max  = min(N, int(T_anim / dt))
idx    = np.arange(0, n_max, step)
trail_n = int(trail_s / (dt * step))

x1, y1 = r1[0, idx], r1[1, idx]
x2, y2 = r2[0, idx], r2[1, idx]

# --- Figura ---
fig, ax = plt.subplots(figsize=(6, 6))
L = (l1 + l2) * 1.1
ax.set(xlim=(-L, L), ylim=(-L, L), aspect='equal', xlabel='x (m)', ylabel='y (m)',
       title='Péndulo doble')
ax.grid(alpha=0.3)

rod1,  = ax.plot([], [], 'k-', lw=2)
rod2,  = ax.plot([], [], 'k-', lw=2)
bob1,  = ax.plot([], [], 'o', color='tab:blue', ms=9)
bob2,  = ax.plot([], [], 'o', color='tab:red',  ms=9)
trail, = ax.plot([], [], '-', color='tab:red', lw=1, alpha=0.6)
ax.plot(0, 0, 'ko', ms=6)                                   # pivote
time_txt = ax.text(0.03, 0.95, '', transform=ax.transAxes)

def update(k):
    rod1.set_data([0, x1[k]], [0, y1[k]])
    rod2.set_data([x1[k], x2[k]], [y1[k], y2[k]])
    bob1.set_data([x1[k]], [y1[k]])
    bob2.set_data([x2[k]], [y2[k]])
    if trail_n > 0:
        j = max(0, k - trail_n)
        trail.set_data(x2[j:k+1], y2[j:k+1])
    time_txt.set_text(f't = {t[idx[k]]:.2f} s')
    return rod1, rod2, bob1, bob2, trail, time_txt

anim = animation(fig, update, frames=len(idx), interval=1000/fps, blit=True)
plt.close(fig)                       # evita mostrar la figura estática duplicada
HTML(anim.to_jshtml())               # reproduce dentro del notebook